In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# CELL [01-SETUP]
!pip install -q lettucedetect groq datasets scikit-learn streamlit
!pip install -q "minicheck @ git+https://github.com/Liyan06/MiniCheck.git@main"
import torch; print("GPU:", torch.cuda.is_available())

In [ ]:
# CELL [B01-INSTALL]
!pip install -q --no-deps lettucedetect
!pip install -q --no-deps "minicheck @ git+https://github.com/Liyan06/MiniCheck.git@main"

In [1]:
# CELL [00-CHECK-ENV]
import numpy, scipy, sklearn, pandas, torch, transformers
print("numpy", numpy.__version__, "| scipy", scipy.__version__, "| sklearn", sklearn.__version__)
print("torch", torch.__version__, "| transformers", transformers.__version__, "| GPU:", torch.cuda.is_available())
from lettucedetect.models.inference import HallucinationDetector; print("lettucedetect OK")
from minicheck.minicheck import MiniCheck; print("minicheck OK")

numpy 2.1.3 | scipy 1.16.3 | sklearn 1.6.1
torch 2.11.0+cu130 | transformers 5.20.2 | GPU: True
lettucedetect OK
minicheck OK


In [2]:
# CELL [DATA-REBUILD]
import os, re, pandas as pd
from nltk.tokenize import PunktSentenceTokenizer
from sklearn.model_selection import GroupShuffleSplit

if not os.path.exists("RAGTruth"):
    !git clone -q https://github.com/ParticleMedia/RAGTruth
resp = pd.read_json("RAGTruth/dataset/response.jsonl", lines=True)
src  = pd.read_json("RAGTruth/dataset/source_info.jsonl", lines=True)
df = resp.merge(src, on="source_id", how="left")

df["has_halluc"] = df["labels"].apply(lambda x: len(x) > 0)
df["context"]    = df["prompt"]
df["question"]   = df.apply(lambda r: r["source_info"]["question"] if r["task_type"]=="QA" else "", axis=1)
df["gold_spans"] = df["labels"].apply(lambda L: [(l["start"], l["end"]) for l in L])

_punkt = PunktSentenceTokenizer()
def split_sents(text):
    spans = []
    for m in re.finditer(r'[^\n]+', text):
        base = m.start()
        for a, b in _punkt.span_tokenize(m.group()):
            if text[base+a:base+b].strip():
                spans.append((base+a, base+b))
    return spans
df["sent_spans"] = df["response"].apply(split_sents)

train_all, test = df[df.split=="train"], df[df.split=="test"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
tr, va = [], []
for task, g in train_all.groupby("task_type"):
    a, b = next(gss.split(g, groups=g["source_id"]))
    tr.append(g.iloc[a]); va.append(g.iloc[b])
train, val = pd.concat(tr), pd.concat(va)

for name, d in [("train", train), ("val", val), ("test", test)]:
    d.to_pickle(f"/kaggle/working/{name}.pkl")
    print(name, len(d), d.groupby("task_type")["has_halluc"].mean().round(3).to_dict())

train 12822 {'Data2txt': 0.693, 'QA': 0.311, 'Summary': 0.315}
val 2268 {'Data2txt': 0.695, 'QA': 0.307, 'Summary': 0.29}
test 2700 {'Data2txt': 0.643, 'QA': 0.178, 'Summary': 0.227}


In [3]:
# CELL [B02-LETTUCE-TEST]
import pandas as pd, time, torch
from transformers import AutoTokenizer
from lettucedetect.models.inference import HallucinationDetector
print("GPU:", torch.cuda.is_available())

val = pd.read_pickle("/kaggle/working/val.pkl")
MODEL = "KRLabsOrg/lettucedect-base-modernbert-en-v1"

# Real token lengths (checks the ~4,096-token limit properly)
tok = AutoTokenizer.from_pretrained(MODEL)
lens = val.apply(lambda r: len(tok(r["context"] + " " + r["question"] + " " + r["response"])["input_ids"]), axis=1)
print("Max real tokens:", lens.max(), "| answers over 4096:", (lens > 4096).sum())

det = HallucinationDetector(method="transformer", model_path=MODEL)
sample = val.groupby("task_type").head(7)          # 21 answers, 7 per task
t = time.time(); preds = []
for _, r in sample.iterrows():
    preds.append(det.predict(context=[r["context"]], question=r["question"],
                             answer=r["response"], output_format="spans"))
print(f"{len(sample)} answers in {time.time()-t:.1f}s")

for (_, r), p in zip(sample.iterrows(), preds):
    if r["has_halluc"]:
        print("\nTASK:", r["task_type"])
        print("PRED:", [(s.get("start"), s.get("end"), s.get("text")) for s in p])
        print("GOLD:", [r["response"][a:b] for a, b in r["gold_spans"]])
        break
print("\nPrediction keys:", next((p[0].keys() for p in preds if p), "no spans predicted"))

GPU: True


config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.58M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Max real tokens: 2443 | answers over 4096: 0


model.safetensors: reconstructing file:   0%|          |  0.00B /  598MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

21 answers in 3.8s

TASK: Data2txt
PRED: [(339, 357, ' is open every day')]
GOLD: ['The restaurant is open every day']

Prediction keys: dict_keys(['start', 'end', 'confidence', 'text'])


In [4]:
# CELL [B03-MINICHECK-TEST]
from minicheck.minicheck import MiniCheck
mc = MiniCheck(model_name="flan-t5-large", cache_dir="./ckpts")

docs, claims, owners = [], [], []
for i, (_, r) in enumerate(sample.iterrows()):
    for a, b in r["sent_spans"]:
        docs.append(r["context"]); claims.append(r["response"][a:b]); owners.append(i)

t = time.time()
labels, probs, _, _ = mc.score(docs=docs, claims=claims)
print(f"{len(claims)} sentences in {time.time()-t:.1f}s")

# Show one hallucinated answer: each sentence's support probability
first = next(i for i, (_, r) in enumerate(sample.iterrows()) if r["has_halluc"])
for o, c, p in zip(owners, claims, probs):
    if o == first: print(f"{p:.2f} | {c[:90]}")

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 3.13GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/560 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.13GB            

model.safetensors: downloading bytes:           |  0.00B            

... (6 outputs left out)

added_tokens.json:   0%|          | 0.00/2.63k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.24k [00:00<?, ?B/s]



Evaluating:   0%|          | 0/140 [00:00<?, ?it/s]

Evaluating:   1%|          | 1/140 [00:01<02:21,  1.02s/it]

Evaluating:   1%|▏         | 2/140 [00:01<01:39,  1.39it/s]

Evaluating:   2%|▏         | 3/140 [00:02<01:24,  1.63it/s]

Evaluating:   3%|▎         | 4/140 [00:02<01:17,  1.75it/s]

Evaluating:   4%|▎         | 5/140 [00:03<01:13,  1.84it/s]

Evaluating:   4%|▍         | 6/140 [00:03<01:10,  1.91it/s]

Evaluating:   5%|▌         | 7/140 [00:04<01:08,  1.93it/s]

Evaluating:   6%|▌         | 8/140 [00:04<01:07,  1.94it/s]

Evaluating:   6%|▋         | 9/140 [00:05<01:07,  1.95it/s]

Evaluating:   7%|▋         | 10/140 [00:05<01:05,  1.98it/s]

Evaluating:   8%|▊         | 11/140 [00:06<01:05,  1.98it/s]

Evaluating:   9%|▊         | 12/140 [00:06<01:05,  1.96it/s]

Evaluating:   9%|▉         | 13/140 [00:07<01:04,  1.96it/s]

Evaluating:  10%|█         | 14/140 [00:07<01:04,  1.95it/s]

Evaluating:  11%|█         | 15/140 [00:08<01:03,  1.96it/s]

Evaluating:  11%|█▏     

140 sentences in 50.8s
0.73 | Dumpling King is a Chinese restaurant located at 966 Embarcadero Del Mar, Ste C, Isla Vist
0.03 | The establishment is popular for its dumplings, with the three-flavor dumplings (chives, p
0.23 | Other offerings include chicken wings, skewers, soups, noodles, and a variety of combo mea
0.26 | The restaurant is open every day, with extended hours on Friday.
0.07 | It offers street and lot parking and accommodates group dining with outdoor seating availa
0.94 | The restaurant also offers takeout services.
0.03 | The Dumpling King has a hipster and divey ambience.
0.49 | However, customers have highlighted some areas for improvement such as the taste of some d
0.72 | Overall, the restaurant has a rating of 3.5 stars.
